# Research 18 — Reused holdout check of categorical interactions

**Hypothesis:** the VIP × engagement and loyalty × VIP gains may also appear on the project's existing stratified 80/20 holdout split (seed 42). This is the same split previously used in `player-segment-tuned.ipynb` for LightGBM parameter selection, so it is reused and not independent validation. Fit three LightGBM models on its 80% partition—base features, base + VIP × engagement, and base + loyalty × VIP—and compare predeclared 80/20 probability mixtures. No weights are tuned on this holdout; no Test rows are read.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def make_features(frame, interaction=None):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    if interaction == "vip_engagement":
        vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
        x["vip_engagement"] = vip + "__" + x["engagement_level"].fillna("__MISSING__").astype(str)
    elif interaction == "loyalty_vip":
        vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
        x["loyalty_vip"] = x["loyalty_tier"].fillna("__MISSING__").astype(str) + "__" + vip
    return x.replace([np.inf, -np.inf], np.nan)


indices = np.arange(len(y))
fit_idx, valid_idx = train_test_split(indices, test_size=0.2, stratify=y, random_state=42)
feature_sets = {
    "base": make_features(train),
    "vip_engagement": make_features(train, interaction="vip_engagement"),
    "loyalty_vip": make_features(train, interaction="loyalty_vip"),
}
predictions = {}
started = perf_counter()
for name, X in feature_sets.items():
    x_fit, x_valid = X.iloc[fit_idx].copy(), X.iloc[valid_idx].copy()
    cat_cols = x_fit.select_dtypes(include=["object", "string"]).columns.tolist()
    for col in cat_cols:
        dtype = pd.CategoricalDtype(categories=sorted(x_fit[col].dropna().unique().tolist()))
        x_fit[col] = x_fit[col].astype(dtype)
        x_valid[col] = x_valid[col].astype(dtype)
    model = LGBMClassifier(
        objective="multiclass",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=42,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx], categorical_feature=cat_cols)
    predictions[name] = model.predict_proba(x_valid)
    print(
        f"{name}: macro F1={f1_score(y[valid_idx], predictions[name].argmax(1), average='macro'):.6f}",
        flush=True,
    )

candidates = {
    "Base LightGBM": predictions["base"],
    "80/20 base + VIP × engagement": 0.8 * predictions["base"]
    + 0.2 * predictions["vip_engagement"],
    "80/20 base + loyalty × VIP": 0.8 * predictions["base"] + 0.2 * predictions["loyalty_vip"],
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    rows.append(
        {
            "candidate": name,
            "macro_f1": f1_score(y[valid_idx], pred, average="macro"),
            "weighted_f1": f1_score(y[valid_idx], pred, average="weighted"),
        }
    )
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-holdout-interaction-results.csv", index=False)
print(
    f"Holdout rows={len(valid_idx):,}; elapsed seconds={perf_counter() - started:.1f}. Test not loaded."
)

base: macro F1=0.749904


vip_engagement: macro F1=0.747067


loyalty_vip: macro F1=0.749464


,candidate,macro_f1,weighted_f1
0,Base LightGBM,0.749904,0.763988
1,80/20 base + VIP × engagement,0.750306,0.764353
2,80/20 base + loyalty × VIP,0.749922,0.763884


Holdout rows=20,800; elapsed seconds=88.3. Test not loaded.


## Decision limits

This is the same 20% holdout used in prior LightGBM hyperparameter selection. Although the interaction and 80/20 mixture weight were specified from other screens, this reuse is selection-biased and is not stronger independent evidence. Use it only as a descriptive comparison; this notebook does not create Test predictions.